# ForenSURE-Net: Model Training Run
This notebook runs the Curriculum Learning SRNet training session against the 20,000 image dataset.

**Hardware Recommendation:** T4 x2 or P100 GPU

In [ ]:
import os

# Step out into a safe directory so we don't delete our own working folder!
os.chdir('/kaggle/working')

# Clean up any previous runs
!rm -rf /kaggle/working/ForenSURE-Net

# Clone the repository
!git clone https://github.com/narendraschahar/ForenSURE-Net.git /kaggle/working/ForenSURE-Net

# Set Python Working Directory permanently
os.chdir('/kaggle/working/ForenSURE-Net')
print('Working Directory Set To:', os.getcwd())

In [ ]:
# Install ONLY the extra packages not already on Kaggle.
# We do NOT install from requirements.txt because it would downgrade torch
# from Kaggle's optimized 2.10+cu128 to an older version, breaking CUDA.
!pip install -q scikit-learn tqdm Pillow

In [ ]:
# ── DATA SANITY CHECK ────────────────────────────────────────────────────────
# Verify stego images are actually DIFFERENT from cover images
import numpy as np
from PIL import Image
import glob

BASE = '/kaggle/input/datasets/narendrachahar/forensure-dataset-20k/ForenSURE_Dataset'

cover_files = sorted(glob.glob(f'{BASE}/cover/*.pgm'))[:5]
print(f'Checking {len(cover_files)} cover/stego pairs...')

for cf in cover_files:
    name = os.path.basename(cf)
    sf_10 = f'{BASE}/stego_hill_10/{name}'
    sf_04 = f'{BASE}/stego_hill_04/{name}'

    cover = np.array(Image.open(cf)).astype(np.float32)

    diff_10, diff_04 = 'N/A', 'N/A'
    if os.path.exists(sf_10):
        diff_10 = int(np.sum(cover != np.array(Image.open(sf_10)).astype(np.float32)))
    if os.path.exists(sf_04):
        diff_04 = int(np.sum(cover != np.array(Image.open(sf_04)).astype(np.float32)))

    print(f'{name}: 1.0bpp changed pixels={diff_10} | 0.4bpp changed pixels={diff_04}')

print('\nIf ALL values are 0 the stego embedding FAILED. Otherwise we are good!')

In [ ]:
# ── DEBUG VERIFICATION RUN ───────────────────────────────────────────────────
# Uses only 500 pairs and 3 epochs to verify the model learns BEFORE full run.
# If ROC AUC > 0.55 after epoch 3, the pipeline is working. Run full training next.
!python scripts/train_residual.py --config configs/kaggle_debug_config.json --debug

In [ ]:
# ── FULL TRAINING: Phase 1 (1.0 bpp) ─────────────────────────────────────────
# Only run this AFTER verifying the debug run shows ROC AUC > 0.55
# !python scripts/train_residual.py --config configs/kaggle_phase1_config.json

In [ ]:
# ── FULL TRAINING: Phase 2 (0.4 bpp) ─────────────────────────────────────────
# !python scripts/train_residual.py --config configs/kaggle_phase2_config.json --resume experiments/checkpoints/srnet_phase1_10bpp.pth